In [2]:
import pandas as pd

df = pd.read_csv('adidas_supply_chain_dataset.csv')

In [4]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Location,Size,Color,Season,Review Rating,...,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases,Purchase Amount (EUR),Warehouse,Order Status,Lead Time (Days),Carrier
0,1,55,Male,Adilette Slides,Clothing,Berlin,L,Gray,Winter,3.1,...,Yes,Yes,14,Visa,Fortnightly,48.76,Herzogenaurach DC,In Transit,4,Hermes
1,2,19,Male,Running T-Shirt,Clothing,Leipzig,L,Maroon,Winter,3.1,...,Yes,Yes,2,Klarna,Fortnightly,58.88,Leipzig DC,Returned,4,Hermes
2,3,50,Male,Ultraboost Shoes,Clothing,Erlangen,S,Maroon,Spring,3.1,...,Yes,Yes,23,SEPA Direct Debit,Weekly,67.16,Herzogenaurach DC,Delivered,5,DPD
3,4,21,Male,Football Jersey,Footwear,Leipzig,M,Maroon,Spring,3.5,...,Yes,Yes,49,Apple Pay,Weekly,82.80,Nuremberg DC,Processing,5,DPD
4,5,45,Male,Football Jersey,Clothing,Stuttgart,M,Turquoise,Spring,2.7,...,Yes,Yes,31,Mastercard,Annually,45.08,Herzogenaurach DC,In Transit,4,DHL


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Location                3900 non-null   object 
 6   Size                    3900 non-null   object 
 7   Color                   3900 non-null   object 
 8   Season                  3900 non-null   object 
 9   Review Rating           3863 non-null   float64
 10  Subscription Status     3900 non-null   object 
 11  Shipping Type           3900 non-null   object 
 12  Discount Applied        3900 non-null   object 
 13  Promo Code Used         3900 non-null   object 
 14  Previous Purchases      3900 non-null   

In [7]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
Purchase Amount (EUR)      0
Warehouse                  0
Order Status               0
Lead Time (Days)           0
Carrier                    0
dtype: int64

In [8]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [9]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
Purchase Amount (EUR)     0
Warehouse                 0
Order Status              0
Lead Time (Days)          0
Carrier                   0
dtype: int64

In [13]:
# Renaming columns according to snake casing for better readability and documentation

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(eur)':'purchase_amount'})

In [14]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'location', 'size', 'color', 'season', 'review_rating',
       'subscription_status', 'shipping_type', 'discount_applied',
       'promo_code_used', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'purchase_amount', 'warehouse',
       'order_status', 'lead_time_(days)', 'carrier'],
      dtype='object')

In [15]:
# create a new column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [16]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [17]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [18]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [19]:
!pip install pyodbc sqlalchemy 

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------------ --------------- 1.3/2.2 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.7 MB/s eta 0:00:00


In [20]:
!pip install pyodbc sqlalchemy 

In [26]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

connection_string = quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=customer_behaviour;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")

# Write DataFrame to SQL Server
df.to_sql("customer", engine, if_exists="replace", index=False)

pd.read_sql("SELECT TOP 5 * FROM customer;", engine)


,customer_id,age,gender,item_purchased,category,location,size,color,season,review_rating,...,previous_purchases,payment_method,frequency_of_purchases,purchase_amount,warehouse,order_status,lead_time_(days),carrier,age_group,purchase_frequency_days
0,1,55,Male,Adilette Slides,Clothing,Berlin,L,Gray,Winter,3.1,...,14,Visa,Fortnightly,48.76,Herzogenaurach DC,In Transit,4,Hermes,Middle-aged,14
1,2,19,Male,Running T-Shirt,Clothing,Leipzig,L,Maroon,Winter,3.1,...,2,Klarna,Fortnightly,58.88,Leipzig DC,Returned,4,Hermes,Young Adult,14
2,3,50,Male,Ultraboost Shoes,Clothing,Erlangen,S,Maroon,Spring,3.1,...,23,SEPA Direct Debit,Weekly,67.16,Herzogenaurach DC,Delivered,5,DPD,Middle-aged,7
3,4,21,Male,Football Jersey,Footwear,Leipzig,M,Maroon,Spring,3.5,...,49,Apple Pay,Weekly,82.80,Nuremberg DC,Processing,5,DPD,Young Adult,7
4,5,45,Male,Football Jersey,Clothing,Stuttgart,M,Turquoise,Spring,2.7,...,31,Mastercard,Annually,45.08,Herzogenaurach DC,In Transit,4,DHL,Middle-aged,365
